In [1]:
# The code in this block comes directly from data_sort_and_split.ipynb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

heart_data = pd.read_csv("heart.csv")

heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})
heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])

categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)

feature_matrix = heart_data.drop("HeartDisease", axis=1)
target_labels = heart_data["HeartDisease"]

features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42,
    stratify= target_labels
)

# Scaling performed to avoid features with larger ranges being seen as more significant to model
# Furthermore, it prevents data leakage by fitting to training data only.

scaler = StandardScaler()

# WWe 'fit' only on training data to prevent data leakage from the test set.
scaler.fit(features_train)

# We transform both sets using the training parameters so the scale is consistent.
features_train_scaled = scaler.transform(features_train)
features_test_scaled = scaler.transform(features_test)


In [2]:
from sklearn.svm import SVC

from sklearn.svm import SVC

# probability=True is REQUIRED so we can call predict_proba() for ROC-AUC later.
# Without it, sklearn will throw an error when you try to compute ROC-AUC.
# kernel='linear' and C=1.0 stay the same as before.
svm_model = SVC(kernel='linear', C=1.0, random_state=42, probability=True)

svm_model.fit(features_train_scaled, targets_train)

predictions = svm_model.predict(features_test_scaled)
predicted_proba = svm_model.predict_proba(features_test_scaled)[:, 1]



In [3]:
#Metric Calculation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy  = accuracy_score(targets_test, predictions)
precision = precision_score(targets_test, predictions)
recall    = recall_score(targets_test, predictions)
f1        = f1_score(targets_test, predictions)
roc_auc   = roc_auc_score(targets_test, predicted_proba)

print("SVM Model Performance:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")

SVM Model Performance:
  Accuracy:  0.8750
  Precision: 0.8559
  Recall:    0.9314
  F1-Score:  0.8920
  ROC-AUC:   0.9309
